# ЛР3. Можно ли доверять LLM в задачах классического машинного обучения

**Курс:** «Методы машинного обучения и искусственного интеллекта»
**Выполнил:** Горбатенков Д. Д.

**Цель:** экспериментально выяснить, в каких задачах классического ML большие языковые модели надёжны, а в каких ошибаются, и оценивать это измеримо (метрики, доверительные интервалы, тесты значимости), а не по впечатлению.

## Что делает ноутбук

1. Фиксирует окружение: `random_state`, версии библиотек (`requirements.txt`), дату и версии моделей.
2. Подключает 4 модели через API или локально: GigaChat, YandexGPT, DeepSeek V4.1 Flash (Yandex AI Studio), Qwen3-4B (Ollama).
3. Собирает банк из 39 задач шести типов (A–F) с эталонными ответами; эталоны перепроверяются кодом.
4. Прогоняет банк через все модели с одинаковым системным промптом и температурой 0. Каждый ответ сразу пишется в `llm_log.jsonl`.
5. Автоматически проверяет типы A, B, E, F. Для C и D выгружает слепую таблицу для ручной проверки.
6. Считает метрики: точность с 95% бутстреп-ДИ, тест Макнемара, ECE, стоимость правильного ответа.
7. Проверяет три гипотезы: **H1, H5, H10**.

## Выбранные гипотезы (по одной из трёх разных групп)

| Группа | Гипотеза | Как проверяем |
|---|---|---|
| 1. Рассуждение или запоминание | **H1. Изменённые числа.** Точность на «учебниковых» задачах выше, чем на тех же задачах с изменёнными числами или условием | 7 пар «учебник ↔ изменённая», парный тест Макнемара |
| 2. Язык, размер, режим | **H5. Размер модели.** Маленькая локальная модель отстаёт от больших сильнее в расчётных задачах (A), чем в концептуальных (B) | разница отставаний (A − B), бутстреп-ДИ |
| 3. Надёжность и доверие | **H10. Калибровка уверенности.** Заявленная уверенность (0–100) систематически завышена | средняя уверенность − точность, ECE, диаграмма надёжности |

Все три гипотезы проверяются по **основному прогону**: дополнительные запросы к API не нужны.

## 0. Прогноз до эксперимента

По правилам ЛР прогноз фиксируется **до** запуска основного прогона коммитом в репозитории; ссылка на коммит приводится в отчёте.

| Гипотеза | Прогноз | Почему так думаю |
|---|---|---|
| H1 | подтвердится | на известных задачах модель может «вспоминать» ответ из обучающих данных |
| H5 | подтвердится | арифметика и многошаговый счёт у маленьких моделей слабее, чем знание фактов |
| H10 | подтвердится | LLM склонны отвечать с уверенностью 90–100 почти всегда |

In [1]:
# 1. Окружение и воспроизводимость
import os, sys, json, re, time, uuid, math, random, datetime, subprocess, tempfile, textwrap, getpass, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import sklearn
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score, accuracy_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

WORKDIR = Path(os.environ.get("LR3_DIR", "lr3_results"))
WORKDIR.mkdir(exist_ok=True)
LOG_PATH = WORKDIR / "llm_log.jsonl"          # все вызовы моделей, дописываются сразу после ответа

RUN_DATE = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print("Дата запуска:", RUN_DATE)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__)
print("Результаты пишутся в:", WORKDIR.resolve())

Дата запуска: 2026-11-05 17:04
Python 3.14.7 | numpy 2.5.4 | pandas 3.1.1 | sklearn 1.9.1
Результаты пишутся в: C:\Users\nnerf\OneDrive\Desktop\ЛР3\lr3_results


In [2]:
# Версии всех библиотек -> requirements.txt (требование воспроизводимости)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(WORKDIR / "requirements.txt").write_text(freeze, encoding="utf-8")
print(f"Сохранено {len(freeze.splitlines())} пакетов в {WORKDIR / 'requirements.txt'}")

Сохранено 116 пакетов в lr3_results\requirements.txt


## 1. Модели и подключение

| Роль | Модель | Доступ |
|---|---|---|
| Российская № 1 | GigaChat-2-Max | GigaChat API (бесплатный лимит для физлиц) |
| Российская № 2 | YandexGPT Pro 5.1 (`yandexgpt-5.1`) | Yandex AI Studio, OpenAI-совместимый API |
| Зарубежная большая | DeepSeek V4.1 Flash (`deepseek-v4.1-flash`) | галерея моделей Yandex AI Studio |
| Маленькая локальная | Qwen3-4B-Instruct-2507 (`qwen3:4b-instruct-2507-q4_K_M`) | Ollama на ноутбуке |

**Ключи** берутся из файла `keys.env` рядом с ноутбуком (по строке `ИМЯ=значение`). Файл внесён в `.gitignore`, поэтому в репозиторий ключи не попадают:
- `GIGACHAT_AUTH_KEY`: «Ключ авторизации» из личного кабинета GigaChat API (уже в base64);
- `YANDEX_API_KEY` и `YANDEX_FOLDER_ID`: API-ключ сервисного аккаунта и ID каталога;

**Цены** (`price_in` и `price_out`, рубли за 1000 токенов) нужно взять с официальных страниц тарифов и записать дату. Цены DeepSeek указаны в долларах, их переводим по курсу ЦБ на дату. Пока цена равна `None`, график «стоимость–точность» эту модель пропускает.

In [3]:
MODELS = {
    "GigaChat-2-Max": dict(backend="gigachat", model_id="GigaChat-2-Max", role="ru_big",
                           price_in=None, price_out=None),
    "YandexGPT-Pro-5.1": dict(backend="yandex", model_id="yandexgpt-5.1", role="ru_big",
                           price_in=None, price_out=None),
    # Зарубежная большая модель из галереи Yandex AI Studio (URI — часть после gpt://<folder_id>/)
    "DeepSeek-V4.1-Flash": dict(backend="yandex", model_id="deepseek-v4.1-flash", role="foreign_big",
                           price_in=None, price_out=None),
    # Версия без режима рассуждений (instruct): большие модели тоже отвечают без него
    "Qwen3-4B-local": dict(backend="ollama",   model_id="qwen3:4b-instruct-2507-q4_K_M", role="small",
                           price_in=0.0, price_out=0.0),
}
SMALL_MODEL = "Qwen3-4B-local"
BIG_MODELS = [m for m, c in MODELS.items() if c["role"] != "small"]

TEMPERATURE = 0.0
MAX_TOKENS = 1500
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
GIGACHAT_VERIFY_SSL = False   # у GigaChat сертификат НУЦ Минцифры; можно указать путь к .crt вместо False

# Ключи хранятся в файле keys.env рядом с ноутбуком (строки вида ИМЯ=значение).
# Файл внесён в .gitignore и в репозиторий не попадает.
KEYS_FILE = Path("keys.env")
if KEYS_FILE.exists():
    for line in KEYS_FILE.read_text(encoding="utf-8-sig").splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
Path(".gitignore").write_text("keys.env\n.venv/\n.idea/\n", encoding="utf-8") if not Path(".gitignore").exists() else None

def get_secret(name):
    """Ключ из keys.env / переменной окружения."""
    if not os.environ.get(name):
        raise RuntimeError(f"Нет ключа {name}: добавьте строку {name}=... в файл {KEYS_FILE.resolve()}")
    return os.environ[name]

print("Ключи найдены:", {k: bool(os.environ.get(k)) for k in ["GIGACHAT_AUTH_KEY", "YANDEX_FOLDER_ID", "YANDEX_API_KEY"]})

Ключи найдены: {'GIGACHAT_AUTH_KEY': True, 'YANDEX_FOLDER_ID': True, 'YANDEX_API_KEY': True}


In [4]:
# CELL: backends. Низкоуровневые вызовы каждого API. Все возвращают (текст, токены_вход, токены_выход, версия_модели)
import urllib3
urllib3.disable_warnings()

_giga_token = {"value": None, "expires": 0}

def _gigachat_token():
    if _giga_token["value"] and time.time() < _giga_token["expires"] - 60:
        return _giga_token["value"]
    r = requests.post("https://ngw.devices.sberbank.ru:9443/api/v2/oauth",
                      headers={"Authorization": f"Basic {get_secret('GIGACHAT_AUTH_KEY')}",
                               "RqUID": str(uuid.uuid4()),
                               "Content-Type": "application/x-www-form-urlencoded"},
                      data={"scope": os.environ.get("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")},
                      verify=GIGACHAT_VERIFY_SSL, timeout=30)
    r.raise_for_status()
    j = r.json()
    _giga_token.update(value=j["access_token"], expires=j["expires_at"] / 1000)
    return _giga_token["value"]

def call_gigachat(model_id, messages, temperature, max_tokens, cfg):
    body = dict(model=model_id, messages=messages, temperature=temperature, max_tokens=max_tokens)
    r = requests.post("https://gigachat.devices.sberbank.ru/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {_gigachat_token()}"},
                      json=body, verify=GIGACHAT_VERIFY_SSL, timeout=180)
    r.raise_for_status()
    j = r.json()
    u = j.get("usage", {})
    return j["choices"][0]["message"]["content"], u.get("prompt_tokens"), u.get("completion_tokens"), j.get("model", model_id)

def _openai_compatible(url, headers, model, messages, temperature, max_tokens):
    r = requests.post(url, headers=headers, timeout=180,
                      json=dict(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens))
    r.raise_for_status()
    j = r.json()
    u = j.get("usage") or {}
    return j["choices"][0]["message"]["content"], u.get("prompt_tokens"), u.get("completion_tokens"), j.get("model", model)

def call_yandex(model_id, messages, temperature, max_tokens, cfg):
    folder = get_secret("YANDEX_FOLDER_ID")
    return _openai_compatible("https://llm.api.cloud.yandex.net/v1/chat/completions",
                              {"Authorization": f"Bearer {get_secret('YANDEX_API_KEY')}", "OpenAI-Project": folder},
                              f"gpt://{folder}/{model_id}", messages, temperature, max_tokens)

def call_deepseek(model_id, messages, temperature, max_tokens, cfg):
    return _openai_compatible("https://api.deepseek.com/chat/completions",
                              {"Authorization": f"Bearer {get_secret('DEEPSEEK_API_KEY')}"},
                              model_id, messages, temperature, max_tokens)

def call_ollama(model_id, messages, temperature, max_tokens, cfg):
    body = dict(model=model_id, messages=messages, stream=False,
                options=dict(temperature=temperature, seed=RANDOM_STATE, num_predict=max_tokens))
    if "think" in cfg:
        body["think"] = cfg["think"]
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=body, timeout=600)
    r.raise_for_status()
    j = r.json()
    text = re.sub(r"<think>.*?</think>", "", j["message"]["content"], flags=re.S).strip()
    return text, j.get("prompt_eval_count"), j.get("eval_count"), j.get("model", model_id)

BACKENDS = {"gigachat": call_gigachat, "yandex": call_yandex, "deepseek": call_deepseek, "ollama": call_ollama}

### Функция `ask(model, prompt, temperature) → (ответ, токены, время)`

Это единственная точка обращения к моделям, как и требует задание. Каждый вызов **сразу** дописывается строкой в `llm_log.jsonl`: дата, модель, версия, промпт, ответ, токены, время.
`ask_cached` поверх неё добавляет кэш по ключу (модель, задача, вариант). При перезапуске ноутбука ответы берутся из лога, повторных запросов нет. Поэтому результаты воспроизводятся, даже если модель на сервере обновится.

In [5]:
SYSTEM_PROMPT = (
    "Ты решаешь задачи по классическому машинному обучению. Рассуждай кратко и по делу. "
    "В самом конце ответа обязательно напиши две отдельные строки:\n"
    "ОТВЕТ: <ответ>\n"
    "УВЕРЕННОСТЬ: <целое число от 0 до 100, насколько ты уверен в ответе>\n"
    "Если в условии недостаточно данных, чтобы получить ответ, напиши: ОТВЕТ: недостаточно данных"
)

def ask(model, prompt, temperature=TEMPERATURE, system=SYSTEM_PROMPT, max_tokens=MAX_TOKENS, meta=None):
    """Один запрос к модели. Возвращает (ответ, токены{in,out}, время_сек). Лог пишется сразу."""
    cfg = MODELS[model]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    err, text, tin, tout, version = None, None, None, None, None
    t0 = time.time()
    for attempt in range(3):                       # до 3 попыток при сетевых ошибках
        try:
            text, tin, tout, version = BACKENDS[cfg["backend"]](cfg["model_id"], messages, temperature, max_tokens, cfg)
            err = None
            break
        except Exception as e:
            err = repr(e)
            time.sleep(2 * (attempt + 1))
    elapsed = time.time() - t0
    rec = dict(ts=datetime.datetime.now().isoformat(timespec="seconds"), model=model, model_id=cfg["model_id"],
               model_version=version, temperature=temperature, system=system, prompt=prompt, response=text,
               tokens_in=tin, tokens_out=tout, seconds=round(elapsed, 3), error=err, **(meta or {}))
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    ask.last_record = rec
    return text, {"in": tin, "out": tout}, elapsed

def load_log():
    if not LOG_PATH.exists():
        return []
    return [json.loads(l) for l in LOG_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

_CACHE = None   # (модель, ключ) -> последняя успешная запись лога

def ask_cached(model, prompt, key, temperature=TEMPERATURE, **kw):
    """Как ask, но если ответ с таким ключом уже есть в логе без ошибки, берём его из лога."""
    global _CACHE
    if _CACHE is None:
        _CACHE = {(r["model"], r.get("key")): r for r in load_log() if r.get("error") is None}
    if (model, key) in _CACHE:
        return _CACHE[(model, key)]
    ask(model, prompt, temperature=temperature, meta={"key": key}, **kw)
    rec = ask.last_record
    if rec.get("error") is None:
        _CACHE[(model, key)] = rec
    return rec

In [6]:
# Проверка подключения ко всем четырём моделям одним тестовым запросом
ping_rows = []
for m in MODELS:
    text, tok, sec = ask(m, "Сколько будет 2+2?", meta={"key": "ping"})
    ping_rows.append(dict(model=m, ok=text is not None, seconds=round(sec, 2), tokens=tok,
                          answer=(text or ask.last_record["error"])[:80]))
pd.DataFrame(ping_rows)

,model,ok,seconds,tokens,answer
0,GigaChat-2-Max,True,4.83,"{'in': 103, 'out': 60}",Хотя это не задача классического машинного обу...
1,YandexGPT-Pro-5.1,True,0.77,"{'in': 98, 'out': 15}",ОТВЕТ: 4\nУВЕРЕННОСТЬ: 100
2,DeepSeek-V4.1-Flash,True,1.68,"{'in': 139, 'out': 43}",ОТВЕТ: 4\nУВЕРЕННОСТЬ: 100
3,Qwen3-4B-local,True,25.94,"{'in': 135, 'out': 23}",ОТВЕТ: 4 \nУВЕРЕННОСТЬ: 100


## 2. Банк задач

Шесть типов задач, темы курса: линейная и логистическая регрессия, kNN, деревья и ансамбли, SVM, байесовские методы, кластеризация, PCA, метрики.

| Тип | Что это | Сколько | Как проверяется |
|---|---|---|---|
| A | расчёт, однозначное число | 11 | автоматически, допуск ±1% относительный |
| B | «верно/неверно» (половина ложных) | 8 | автоматически по вердикту |
| C | выбор метода под инженерную задачу | 5 | вручную по рубрике 0–2, вслепую |
| D | генерация кода на sklearn | 5 | запуск без правок + чек-лист методологии |
| E | LLM как классификатор (breast_cancer) | 1 эксперимент × 100 строк | F1 редкого класса, balanced accuracy |
| F | ловушки: изменённые числа, недостающие данные, лишние данные | 10 | автоматически |

Для H1 используются **7 пар**: «учебниковая» задача (A01–A07) и её версия с изменёнными числами или условием (F01–F07), связанные полем `pair`.

Эталоны задач A и F решены вручную и записаны в банк числами (4 знака после запятой). Следующая ячейка пересчитывает каждый эталон кодом и проверяет совпадение. Эталоны задач D — результат референсного решения.

In [ ]:
def T(id, type, topic, ru, etalon, source, difficulty, tol=0.01, pair=None, kind=None, en=None, split="eval"):
    return dict(id=id, type=type, topic=topic, text_ru=ru, text_en=en, etalon=etalon, tol=tol,
                source=source, difficulty=difficulty, pair=pair, kind=kind, split=split)

TASKS = [
    # ---------- Тип A: расчётные (A01–A07 — «учебниковые», у каждой есть пара в F) ----------
    T("A01", "A", "байес", "Датчик неисправности срабатывает в 95% случаев при наличии дефекта и в 5% случаев без него. "
      "Дефект есть у 1% деталей. Датчик сработал. Какова вероятность дефекта?", 0.1610, "учебник", 2, pair="p1", kind="учебник"),
    T("A02", "A", "деревья", "В узле дерева 40 объектов класса 0 и 10 класса 1. Разбиение даёт левый узел (35; 2) и правый (5; 8) "
      "(числа объектов классов 0 и 1). Найдите прирост информации по критерию Джини.", 0.1212, "учебник", 2, pair="p2", kind="учебник"),
    T("A03", "A", "метрики", "Матрица ошибок: TP = 30, FP = 10, FN = 20, TN = 940. Найдите F1-меру.", 0.6667, "учебник", 1, pair="p3", kind="учебник"),
    T("A04", "A", "SVM", "Для разделяющей гиперплоскости линейного SVM w = (3; 4), b = −2. Найдите ширину зазора (margin) между "
      "гиперплоскостями w·x + b = ±1.", 0.4, "учебник", 1, pair="p4", kind="учебник"),
    T("A05", "A", "лин. регрессия", "Дана выборка: x = [1, 2, 3, 4, 5], y = [2, 4, 5, 4, 5]. Найдите коэффициент наклона прямой, "
      "построенной методом наименьших квадратов.", 0.6, "учебник", 1, pair="p5", kind="учебник"),
    T("A06", "A", "PCA", "Ковариационная матрица двух признаков равна [[2, 1], [1, 2]]. Какую долю дисперсии объясняет первая "
      "главная компонента?", 0.75, "учебник", 2, pair="p6", kind="учебник"),
    T("A07", "A", "ансамбли", "В бэггинге строится бутстреп-выборка того же размера n, что и исходная. К какому значению стремится "
      "доля объектов, не попавших в бутстреп-выборку, при n → ∞?", 0.3679, "учебник", 2, pair="p7", kind="учебник"),
    T("A08", "A", "лог. регрессия", "Логистическая регрессия: w = (0,5; −1,2), b = 0,3. Найдите вероятность класса 1 для объекта "
      "x = (2; 1).", 0.5250, "придумана", 1),
    T("A09", "A", "kNN", "Обучающая выборка (x → y): 1 → 3, 2 → 5, 4 → 4, 7 → 10, 8 → 12. Предскажите y для x = 5,4 методом "
      "kNN-регрессии с k = 3 (среднее значение соседей, евклидово расстояние).", 8.6667, "придумана", 2),
    T("A10", "A", "деревья", "В узле 8 объектов класса «+» и 8 класса «−». Разбиение даёт левый узел (6+, 2−) и правый (2+, 6−). "
      "Найдите прирост информации по энтропии (логарифм по основанию 2).", 0.1887, "придумана", 2),
    T("A11", "A", "кластеризация", "Точки: (0;0), (0;2), (4;0), (4;2), (10;1). Начальные центроиды k-means: c1 = (0;1), c2 = (5;1). "
      "Выполните один шаг (назначение точек + пересчёт центроидов). Чему равна координата x нового центроида c2?",
      6.0, "придумана", 2),

    # ---------- Тип B: верно / неверно (4 верных, 4 ложных правдоподобных) ----------
    T("B01", "B", "ансамбли/SVM", "Утверждение: «Случайный лес не требует масштабирования признаков, а SVM с RBF-ядром требует». "
      "Верно или неверно? Обоснуйте.", "верно", "учебник", 1),
    T("B02", "B", "байес", "Утверждение: «Наивный Байес даёт хорошо калиброванные вероятности, потому что он вероятностный». "
      "Верно или неверно? Обоснуйте.", "неверно", "учебник", 2),
    T("B03", "B", "PCA", "Утверждение: «PCA выбирает направления, лучше всего разделяющие классы». Верно или неверно? Обоснуйте.",
      "неверно", "учебник", 1),
    T("B04", "B", "лин. регрессия", "Утверждение: «L1-регуляризация в линейной регрессии может обнулять коэффициенты, "
      "а L2-регуляризация, как правило, нет». Верно или неверно? Обоснуйте.", "верно", "придумана", 1),
    T("B05", "B", "метрики", "Утверждение: «При соотношении классов 99:1 accuracy — подходящая основная метрика качества "
      "классификатора». Верно или неверно? Обоснуйте.", "неверно", "придумана", 1),
    T("B06", "B", "ансамбли", "Утверждение: «Бэггинг обучает базовые модели на бутстреп-выборках и в основном снижает дисперсию "
      "ансамбля, а не смещение». Верно или неверно? Обоснуйте.", "верно", "придумана", 2),
    T("B07", "B", "кластеризация", "Утверждение: «Алгоритм k-means при любой инициализации гарантированно находит глобальный "
      "минимум суммы квадратов расстояний до центроидов». Верно или неверно? Обоснуйте.", "неверно", "придумана", 1),
    T("B08", "B", "лог. регрессия", "Утверждение: «Логистическая регрессия — линейный классификатор: её разделяющая граница в "
      "исходном пространстве признаков — гиперплоскость». Верно или неверно? Обоснуйте.", "верно", "придумана", 2),

    # ---------- Тип C: выбор метода (эталон — рубрика) ----------
    T("C01", "C", "выбор метода", "Есть 150 размеченных записей вибросигнала, 400 спектральных признаков, 3 класса состояния "
      "подшипника. Нужно интерпретируемое решение. Какой метод выбрать и почему?",
      "2 балла: интерпретируемая модель с сильной регуляризацией/отбором признаков (L1-логрегрессия, неглубокое дерево) + "
      "упомянута опасность p≫n и нужна стратифицированная кросс-валидация. 1: метод разумен, но без учёта p≫n или "
      "интерпретируемости. 0: глубокие нейросети/неинтерпретируемый метод без обоснования.", "учебник", 2),
    T("C02", "C", "выбор метода", "Банк: 1 млн транзакций, мошеннических 0,5%, 30 табличных признаков. Нужно ранжировать "
      "транзакции по риску для ручной проверки. Какой метод и какую метрику выбрать?",
      "2: градиентный бустинг или логрегрессия с весами классов + метрика PR-AUC/precision@k (не accuracy) + разбиение с "
      "учётом времени или стратификацией. 1: метод верный, метрика или валидация не обоснованы. 0: accuracy как метрика.",
      "придумана", 2),
    T("C03", "C", "выбор метода", "Нужно сегментировать 5000 клиентов магазина без разметки по 20 числовым признакам в разных "
      "единицах (рубли, дни, штуки). Какой метод выбрать и что сделать до него?",
      "2: масштабирование признаков + k-means/GMM (или DBSCAN) + выбор числа кластеров (силуэт/локоть). 1: метод "
      "кластеризации без масштабирования или без выбора k. 0: метод обучения с учителем.", "придумана", 1),
    T("C04", "C", "выбор метода", "2000 писем, разметка спам/не спам, признаки — мешок слов (20 000 слов). Нужен быстрый и "
      "сильный базовый классификатор. Что выбрать?",
      "2: мультиномиальный наивный Байес или линейный SVM/логрегрессия на TF-IDF + объяснение (разреженные признаки, мало "
      "данных). 1: метод подходит, обоснование слабое. 0: kNN/дерево без обоснования для разреженных данных.", "придумана", 1),
    T("C05", "C", "выбор метода", "Предсказание цены квартиры по 15 признакам, зависимость местами нелинейная, заказчику нужно "
      "понимать важность признаков. 8000 объектов. Что выбрать и как оценивать?",
      "2: случайный лес/градиентный бустинг + permutation importance/SHAP + метрика MAE/RMSE на отложенной выборке/КВ. "
      "1: модель верна, без оценки важности или метрики. 0: классификатор или метод без обоснования.", "придумана", 2),

    # ---------- Тип D: код на sklearn (эталон считается референсным решением ниже) ----------
    T("D01", "D", "SVM", "На датасете sklearn load_breast_cancer подбери C для SVM с RBF-ядром на кросс-валидации и выведи "
      "accuracy на отложенной выборке (25%).", None, "учебник", 2, tol=0.03),
    T("D02", "D", "лин. регрессия", "На датасете sklearn load_diabetes подбери коэффициент регуляризации Ridge на кросс-валидации "
      "и выведи R² на отложенной выборке (25%).", None, "придумана", 2, tol=0.06),
    T("D03", "D", "kNN", "На датасете sklearn load_wine подбери число соседей k для kNN-классификатора на кросс-валидации и "
      "выведи accuracy на отложенной выборке (25%).", None, "придумана", 2, tol=0.05),
    T("D04", "D", "метрики", "Сгенерируй данные make_classification(n_samples=2000, weights=[0.95], random_state=42). Обучи "
      "логистическую регрессию и выведи F1-меру для редкого класса на отложенной выборке (25%).", None, "придумана", 2, tol=0.12),
    T("D05", "D", "кластеризация", "На датасете sklearn load_iris выполни кластеризацию KMeans с 3 кластерами и выведи "
      "adjusted Rand index относительно истинных меток.", None, "придумана", 1, tol=0.05),

    # ---------- Тип F: ловушки ----------
    # F01–F07 — те же задачи, что A01–A07, но с изменёнными числами/условием (для H1)
    T("F01", "F", "байес", "Датчик неисправности срабатывает в 90% случаев при наличии дефекта и в 2% случаев без него. "
      "Дефект есть у 3% деталей. Датчик сработал. Какова вероятность дефекта?", 0.5819, "придумана", 2, pair="p1", kind="изменённая"),
    T("F02", "F", "деревья", "В узле дерева 30 объектов класса 0 и 20 класса 1. Разбиение даёт левый узел (25; 5) и правый (5; 15) "
      "(числа объектов классов 0 и 1). Найдите прирост информации по критерию Джини.", 0.1633, "придумана", 2, pair="p2", kind="изменённая"),
    T("F03", "F", "метрики", "Матрица ошибок: TP = 45, FP = 15, FN = 5, TN = 435. Найдите F1-меру.", 0.8182, "придумана", 1,
      pair="p3", kind="изменённая"),
    T("F04", "F", "SVM", "Для разделяющей гиперплоскости линейного SVM w = (5; 12), b = 1. Найдите ширину зазора (margin) между "
      "гиперплоскостями w·x + b = ±1.", 0.1538, "придумана", 1, pair="p4", kind="изменённая"),
    T("F05", "F", "лин. регрессия", "Дана выборка: x = [1, 2, 3, 4, 5], y = [5, 3, 4, 1, 2]. Найдите коэффициент наклона прямой, "
      "построенной методом наименьших квадратов.", -0.8, "придумана", 1, pair="p5", kind="изменённая"),
    T("F06", "F", "PCA", "Ковариационная матрица двух признаков равна [[4, 2], [2, 1]]. Какую долю дисперсии объясняет первая "
      "главная компонента?", 1.0, "придумана", 2, pair="p6", kind="изменённая"),
    T("F07", "F", "ансамбли", "В бэггинге из выборки размера n = 10 строится бутстреп-выборка того же размера. Какова вероятность, "
      "что конкретный объект не попадёт в бутстреп-выборку?", 0.3487, "придумана", 2, pair="p7", kind="изменённая"),
    # недостающие данные — правильный ответ «нельзя определить»
    T("F08", "F", "байес", "Датчик неисправности срабатывает в 95% случаев при наличии дефекта и в 5% случаев без него. "
      "Датчик сработал. Какова вероятность того, что дефект действительно есть?", "НЕЛЬЗЯ", "придумана", 2, kind="недостающие"),
    T("F09", "F", "метрики", "Классификатор на тестовой выборке дал TP = 30 и FP = 10. Найдите recall (полноту).", "НЕЛЬЗЯ",
      "придумана", 1, kind="недостающие"),
    # лишние отвлекающие данные
    T("F10", "F", "SVM", "Линейный SVM обучен на 500 объектах с параметром C = 10, точность на тесте 93%, обучение заняло 2,4 с. "
      "Для разделяющей гиперплоскости w = (6; 8), b = 1. Найдите ширину зазора между гиперплоскостями w·x + b = ±1.",
      0.2, "придумана", 2, kind="лишние"),

    # ---------- 5 задач для отладки промпта (в оценку НЕ входят) ----------
    T("X01", "A", "метрики", "TP = 8, FP = 2. Найдите precision.", 0.8, "придумана", 1, split="debug"),
    T("X02", "B", "kNN", "Утверждение: «kNN требует масштабирования признаков». Верно или неверно? Обоснуйте.", "верно",
      "придумана", 1, split="debug"),
    T("X03", "A", "SVM", "w = (1; 0), b = 0. Найдите ширину зазора.", 2.0, "придумана", 1, split="debug"),
    T("X04", "F", "метрики", "Известно только TN = 50. Найдите accuracy.", "НЕЛЬЗЯ", "придумана", 1, split="debug"),
    T("X05", "B", "деревья", "Утверждение: «Глубокое дерево решений без ограничений склонно к переобучению». Верно или неверно?",
      "верно", "придумана", 1, split="debug"),
]
# Поле text_en из шаблона задания: английская формулировка каждой задачи
EN = {
 "A01": "A fault sensor fires in 95% of cases when a defect is present and in 5% of cases without it. 1% of parts are defective. The sensor fired. What is the probability of a defect?",
 "A02": "A tree node has 40 objects of class 0 and 10 of class 1. A split gives a left node (35; 2) and a right node (5; 8) (counts of classes 0 and 1). Find the information gain by the Gini criterion.",
 "A03": "Confusion matrix: TP = 30, FP = 10, FN = 20, TN = 940. Find the F1 score.",
 "A04": "For a linear SVM separating hyperplane w = (3; 4), b = −2. Find the margin width between the hyperplanes w·x + b = ±1.",
 "A05": "Data: x = [1, 2, 3, 4, 5], y = [2, 4, 5, 4, 5]. Find the slope of the least-squares line.",
 "A06": "The covariance matrix of two features is [[2, 1], [1, 2]]. What share of variance is explained by the first principal component?",
 "A07": "In bagging, a bootstrap sample of the same size n as the original is drawn. What value does the share of objects not included in the bootstrap sample tend to as n → ∞?",
 "A08": "Logistic regression: w = (0.5; −1.2), b = 0.3. Find the probability of class 1 for x = (2; 1).",
 "A09": "Training set (x → y): 1 → 3, 2 → 5, 4 → 4, 7 → 10, 8 → 12. Predict y for x = 5.4 with kNN regression, k = 3 (mean of neighbours, Euclidean distance).",
 "A10": "A node has 8 objects of class '+' and 8 of class '−'. A split gives a left node (6+, 2−) and a right node (2+, 6−). Find the information gain by entropy (log base 2).",
 "A11": "Points: (0;0), (0;2), (4;0), (4;2), (10;1). Initial k-means centroids: c1 = (0;1), c2 = (5;1). Perform one step (assignment + centroid update). What is the x-coordinate of the new centroid c2?",
 "B01": "Statement: 'Random forest does not require feature scaling, whereas an RBF-kernel SVM does.' True or false? Justify.",
 "B02": "Statement: 'Naive Bayes gives well-calibrated probabilities because it is a probabilistic model.' True or false? Justify.",
 "B03": "Statement: 'PCA selects the directions that best separate the classes.' True or false? Justify.",
 "B04": "Statement: 'L1 regularization in linear regression can set coefficients exactly to zero, while L2 regularization typically does not.' True or false? Justify.",
 "B05": "Statement: 'With a 99:1 class ratio, accuracy is a suitable primary metric for a classifier.' True or false? Justify.",
 "B06": "Statement: 'Bagging trains base models on bootstrap samples and mainly reduces the variance of the ensemble rather than its bias.' True or false? Justify.",
 "B07": "Statement: 'k-means is guaranteed to find the global minimum of the within-cluster sum of squares for any initialization.' True or false? Justify.",
 "B08": "Statement: 'Logistic regression is a linear classifier: its decision boundary in the original feature space is a hyperplane.' True or false? Justify.",
 "C01": "There are 150 labelled vibration-signal records, 400 spectral features, 3 bearing condition classes. An interpretable solution is required. Which method would you choose and why?",
 "C02": "A bank has 1 million transactions, 0.5% are fraudulent, 30 tabular features. Transactions must be ranked by risk for manual review. Which method and which metric would you choose?",
 "C03": "5000 store customers must be segmented without labels using 20 numeric features in different units (roubles, days, items). Which method would you choose and what must be done before it?",
 "C04": "2000 e-mails labelled spam/not spam, features are bag-of-words (20,000 words). A fast and strong baseline classifier is needed. What would you choose?",
 "C05": "Predict apartment prices from 15 features; the dependence is partly nonlinear; the client needs to understand feature importance. 8000 objects. What would you choose and how would you evaluate it?",
 "D01": "On sklearn's load_breast_cancer, tune C for an RBF-kernel SVM with cross-validation and print accuracy on a held-out set (25%).",
 "D02": "On sklearn's load_diabetes, tune the Ridge regularization coefficient with cross-validation and print R² on a held-out set (25%).",
 "D03": "On sklearn's load_wine, tune the number of neighbours k for a kNN classifier with cross-validation and print accuracy on a held-out set (25%).",
 "D04": "Generate data with make_classification(n_samples=2000, weights=[0.95], random_state=42). Train logistic regression and print the F1 score for the rare class on a held-out set (25%).",
 "D05": "On sklearn's load_iris, run KMeans with 3 clusters and print the adjusted Rand index with respect to the true labels.",
 "F01": "A fault sensor fires in 90% of cases when a defect is present and in 2% of cases without it. 3% of parts are defective. The sensor fired. What is the probability of a defect?",
 "F02": "A tree node has 30 objects of class 0 and 20 of class 1. A split gives a left node (25; 5) and a right node (5; 15) (counts of classes 0 and 1). Find the information gain by the Gini criterion.",
 "F03": "Confusion matrix: TP = 45, FP = 15, FN = 5, TN = 435. Find the F1 score.",
 "F04": "For a linear SVM separating hyperplane w = (5; 12), b = 1. Find the margin width between the hyperplanes w·x + b = ±1.",
 "F05": "Data: x = [1, 2, 3, 4, 5], y = [5, 3, 4, 1, 2]. Find the slope of the least-squares line.",
 "F06": "The covariance matrix of two features is [[4, 2], [2, 1]]. What share of variance is explained by the first principal component?",
 "F07": "In bagging, a bootstrap sample of the same size is drawn from a sample of size n = 10. What is the probability that a given object is not included in the bootstrap sample?",
 "F08": "A fault sensor fires in 95% of cases when a defect is present and in 5% of cases without it. The sensor fired. What is the probability that a defect is actually present?",
 "F09": "On a test set a classifier gave TP = 30 and FP = 10. Find recall.",
 "F10": "A linear SVM was trained on 500 objects with C = 10; test accuracy is 93%; training took 2.4 s. The separating hyperplane is w = (6; 8), b = 1. Find the margin width between the hyperplanes w·x + b = ±1.",
 "X01": "TP = 8, FP = 2. Find precision.",
 "X02": "Statement: 'kNN requires feature scaling.' True or false? Justify.",
 "X03": "w = (1; 0), b = 0. Find the margin width.",
 "X04": "Only TN = 50 is known. Find accuracy.",
 "X05": "Statement: 'An unrestricted deep decision tree tends to overfit.' True or false?",
}
for t in TASKS:
    t["text_en"] = EN[t["id"]]
tasks = pd.DataFrame(TASKS)
EVAL = tasks[tasks.split == "eval"].reset_index(drop=True)
print("Задач в оценке:", len(EVAL), "| для отладки:", (tasks.split == "debug").sum())
pd.crosstab(EVAL.topic, EVAL.type, margins=True)

In [ ]:
# Проверка эталонов: эталоны в банке решены вручную и записаны числами,
# а здесь каждый пересчитывается кодом, чтобы исключить арифметическую ошибку
def bayes_posterior(sens, fpr, prior):
    """P(дефект | сработал) = P(+|D)P(D) / [P(+|D)P(D) + P(+|не D)P(не D)]"""
    return sens * prior / (sens * prior + fpr * (1 - prior))

def gini(counts):
    p = np.array(counts) / sum(counts)
    return 1 - np.sum(p ** 2)

def entropy(counts):
    p = np.array(counts) / sum(counts)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def split_gain(parent, left, right, impurity):
    n = sum(parent)
    return impurity(parent) - sum(left) / n * impurity(left) - sum(right) / n * impurity(right)

def f1_from_cm(tp, fp, fn):
    p, r = tp / (tp + fp), tp / (tp + fn)
    return 2 * p * r / (p + r)

def svm_margin(w):
    return 2 / np.linalg.norm(w)                     # ширина зазора = 2 / ||w||

def ols_slope(x, y):
    x, y = np.array(x, float), np.array(y, float)
    return np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)

def pca_first_share(cov):
    ev = np.linalg.eigvalsh(np.array(cov, float))
    return ev.max() / ev.sum()                        # доля дисперсии первой главной компоненты

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def knn_regress(xs, ys, q, k):
    idx = np.argsort(np.abs(np.array(xs) - q))[:k]
    return np.mean(np.array(ys)[idx])

def kmeans_one_step(points, centroids):
    P, C = np.array(points, float), np.array(centroids, float)
    lab = np.argmin(((P[:, None, :] - C[None, :, :]) ** 2).sum(-1), axis=1)
    return np.array([P[lab == j].mean(0) for j in range(len(C))])

CHECK = {
    "A01": bayes_posterior(0.95, 0.05, 0.01),
    "A02": split_gain([40, 10], [35, 2], [5, 8], gini),
    "A03": f1_from_cm(30, 10, 20),
    "A04": svm_margin([3, 4]),
    "A05": ols_slope([1, 2, 3, 4, 5], [2, 4, 5, 4, 5]),
    "A06": pca_first_share([[2, 1], [1, 2]]),
    "A07": math.exp(-1),                              # предел (1 - 1/n)^n
    "A08": sigmoid(0.5 * 2 + (-1.2) * 1 + 0.3),
    "A09": knn_regress([1, 2, 4, 7, 8], [3, 5, 4, 10, 12], 5.4, 3),
    "A10": split_gain([8, 8], [6, 2], [2, 6], entropy),
    "A11": kmeans_one_step([(0, 0), (0, 2), (4, 0), (4, 2), (10, 1)], [(0, 1), (5, 1)])[1][0],
    "F01": bayes_posterior(0.90, 0.02, 0.03),
    "F02": split_gain([30, 20], [25, 5], [5, 15], gini),
    "F03": f1_from_cm(45, 15, 5),
    "F04": svm_margin([5, 12]),
    "F05": ols_slope([1, 2, 3, 4, 5], [5, 3, 4, 1, 2]),
    "F06": pca_first_share([[4, 2], [2, 1]]),
    "F07": (1 - 1 / 10) ** 10,
    "F10": svm_margin([6, 8]),
}
etalon_by_id = dict(zip(tasks.id, tasks.etalon))
check = pd.DataFrame([dict(id=k, эталон_вручную=etalon_by_id[k], пересчёт_кодом=round(v, 5),
                           совпадает=abs(etalon_by_id[k] - v) <= 0.001 * max(1, abs(v))) for k, v in CHECK.items()])
assert check.совпадает.all(), check[~check.совпадает]
check

In [ ]:
# Референсные решения задач типа D: их результат и есть эталон для сравнения с кодом модели
from sklearn.datasets import load_breast_cancer, load_diabetes, load_wine, load_iris, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, adjusted_rand_score

def ref_D01():
    X, y = load_breast_cancer(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    gs = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),      # масштабирование ВНУТРИ пайплайна
                      {"svc__C": [0.1, 1, 10, 100]}, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
    gs.fit(Xtr, ytr)                                                           # подбор только на train
    return accuracy_score(yte, gs.predict(Xte))

def ref_D02():
    X, y = load_diabetes(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))).fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

def ref_D03():
    X, y = load_wine(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    gs = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                      {"kneighborsclassifier__n_neighbors": list(range(1, 21))},
                      cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)).fit(Xtr, ytr)
    return accuracy_score(yte, gs.predict(Xte))

def ref_D04():
    X, y = make_classification(n_samples=2000, weights=[0.95], random_state=42)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
    return f1_score(yte, m.predict(Xte), pos_label=1)                          # редкий класс = 1

def ref_D05():
    X, y = load_iris(return_X_y=True)
    lab = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    return adjusted_rand_score(y, lab)

for tid, fn in [("D01", ref_D01), ("D02", ref_D02), ("D03", ref_D03), ("D04", ref_D04), ("D05", ref_D05)]:
    EVAL.loc[EVAL.id == tid, "etalon"] = fn()
    tasks.loc[tasks.id == tid, "etalon"] = EVAL.loc[EVAL.id == tid, "etalon"].values
EVAL[EVAL.type == "D"][["id", "topic", "etalon", "tol"]]

In [ ]:
# Сохраняем банк в формате из задания (JSONL + CSV)
tasks.to_json(WORKDIR / "tasks.jsonl", orient="records", lines=True, force_ascii=False)
tasks.to_csv(WORKDIR / "tasks.csv", index=False)
EVAL.groupby("type").size().rename("задач").to_frame()

## 3. Промпт и отладка формата

Системный промпт **один для всех моделей**. К пользовательскому сообщению добавляется короткая инструкция по формату для своего типа задачи. Промпт отлаживался на 5 задачах X01–X05, которые в оценку не входят. Если подбирать промпт на тех же задачах, на которых оцениваем, получится аналог подгонки гиперпараметров на тесте (вопрос самоконтроля № 10).

In [ ]:
TYPE_SUFFIX = {
    "A": "В строке ОТВЕТ укажи только одно число в виде десятичной дроби (без процентов и единиц).",
    "F": "В строке ОТВЕТ укажи только одно число в виде десятичной дроби (без процентов и единиц).",
    "B": "Сначала кратко обоснуй, затем в строке ОТВЕТ напиши одно слово: верно или неверно.",
    "C": "Назови метод и обоснуй выбор (3–6 предложений). В строке ОТВЕТ кратко назови выбранный метод.",
    "D": ("Напиши полный исполняемый код на Python (sklearn) в одном блоке ```python ... ```. "
          "Последней строкой код должен печатать итоговую метрику в формате: print('РЕЗУЛЬТАТ:', значение). "
          "В строке ОТВЕТ напиши: код выше."),
}

def build_prompt(task):
    return f"{task['text_ru']}\n\n{TYPE_SUFFIX[task['type']]}"

for _, t in tasks[tasks.split == "debug"].iterrows():
    for m in MODELS:
        rec = ask_cached(m, build_prompt(t), key=f"{t['id']}|base")
        tail = (rec["response"] or rec["error"] or "")[-160:].replace("\n", " ⏎ ")
        print(f"{t['id']} | {m:15s} | …{tail}")

## 4. Основной прогон

Все задачи банка × все модели, температура 0, одинаковые промпты. Если ячейку перезапустить, ответы возьмутся из лога: дозапрашиваются только отсутствующие или упавшие с ошибкой.

In [ ]:
rows = []
for _, t in EVAL.iterrows():
    for m in MODELS:
        rec = ask_cached(m, build_prompt(t), key=f"{t['id']}|base")
        rows.append(dict(task_id=t["id"], model=m, response=rec["response"], error=rec["error"],
                         tokens_in=rec["tokens_in"], tokens_out=rec["tokens_out"], seconds=rec["seconds"],
                         model_version=rec["model_version"], ts=rec["ts"]))
raw = pd.DataFrame(rows)
raw.to_csv(WORKDIR / "raw_answers.csv", index=False)
print("Ответов:", len(raw), "| ошибок API:", raw.error.notna().sum())
raw.groupby("model")[["model_version", "ts"]].agg(["first"])

## 5. Разбор и проверка ответов

- **Число** берётся из последней строки `ОТВЕТ:` и приводится к одному виду: «16%», «0,16» и «0.161» → 0.16…; дробь «2/5» → 0.4. Верно, если |ответ − эталон| ≤ 1% от эталона. Если модель написала проценты без знака (16.1 при эталоне 0.161), это тоже засчитывается.
- **Отказ** («недостаточно данных», «нельзя определить» и т. п.) считается неверным ответом, кроме задач с недостающими данными: там отказ и есть правильный ответ. Доля отказов считается отдельно.
- **Вердикт** B: «неверно/ложно» → неверно, «верно/истинно» → верно.
- **Уверенность** берётся из строки `УВЕРЕННОСТЬ:`.

In [ ]:
REFUSAL_RE = re.compile(r"недостаточно|нельзя определить|невозможно (определить|найти|вычислить|ответить)|не хватает|"
                        r"не могу|нет данных|cannot be determined|insufficient", re.I)

def answer_line(text):
    if not isinstance(text, str) or not text:
        return None
    m = re.findall(r"ОТВЕТ\s*[:：]\s*(.+)", text)
    return m[-1].strip().strip("*").strip() if m else None

def parse_confidence(text):
    m = re.findall(r"УВЕРЕННОСТЬ\s*[:：]\s*\**\s*(\d{1,3})", text if isinstance(text, str) else "")
    return min(int(m[-1]), 100) if m else np.nan

def parse_number(s):
    if s is None:
        return None
    s = s.replace("−", "-").replace(" ", "")
    frac = re.search(r"(-?\d+(?:[.,]\d+)?)/(\d+(?:[.,]\d+)?)", s)
    if frac:
        return float(frac.group(1).replace(",", ".")) / float(frac.group(2).replace(",", "."))
    num = re.search(r"-?\d+(?:[.,]\d+)?", s)
    if not num:
        return None
    x = float(num.group(0).replace(",", "."))
    return x / 100 if "%" in s else x

def is_refusal(text):
    text = text if isinstance(text, str) else ""
    a = answer_line(text)
    return bool(REFUSAL_RE.search(a if a is not None else (text or "")[-300:]))

def check_number(x, etalon, tol):
    if x is None:
        return False
    close = lambda v: abs(v - etalon) <= tol * abs(etalon) if etalon != 0 else abs(v) < 1e-3
    return close(x) or (abs(etalon) <= 1 and abs(x) > 1 and close(x / 100))   # проценты без знака

def parse_verdict(s):
    s = (s or "").lower()
    if re.search(r"неверн|ложн|false|не верно", s):
        return "неверно"
    if re.search(r"верн|истинн|true", s):
        return "верно"
    return None

def grade_auto(task, text):
    """Возвращает (верно?, отказ?, распознанный ответ). Для C и D -> верно = NaN (проверка отдельно)."""
    refused = is_refusal(text)
    a = answer_line(text)
    if task["type"] in ("A", "F"):
        if task["etalon"] == "НЕЛЬЗЯ":
            return float(refused), refused, a
        x = None if refused else parse_number(a)
        return float(check_number(x, float(task["etalon"]), task["tol"])), refused, x
    if task["type"] == "B":
        v = parse_verdict(a)
        return float(v == task["etalon"]), refused, v
    return np.nan, refused, a

task_by_id = EVAL.set_index("id").to_dict("index")
res = raw.copy()
graded = [grade_auto({**task_by_id[r.task_id], "id": r.task_id}, r.response) for r in res.itertuples()]
res["correct"] = [g[0] for g in graded]
res["refused"] = [g[1] for g in graded]
res["parsed"] = [g[2] for g in graded]
res["confidence"] = res.response.map(parse_confidence)
res = res.merge(EVAL[["id", "type", "topic", "pair", "kind", "etalon"]], left_on="task_id", right_on="id").drop(columns="id")
res[res.type.isin(["A", "B", "F"])].pivot_table(index="task_id", columns="model", values="correct")

### 5.1. Тип D: запуск сгенерированного кода «как есть»

Код из блока ```python``` запускается отдельным процессом без правок (таймаут 180 с).
- **pass@1**: код выполнился без ошибок с первой попытки.
- **Корректность результата**: напечатанное `РЕЗУЛЬТАТ:` отличается от эталона не больше чем на допуск. Допуск сделан абсолютным, потому что модели могут разбить данные иначе.
- **Методологические ошибки** проверяются вручную по чек-листу в слепой таблице (п. 5.2).

⚠️ Сгенерированный код выполняется локально, поэтому перед запуском он просматривается вручную.

In [ ]:
def extract_code(text):
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text if isinstance(text, str) else "", flags=re.S)
    return max(blocks, key=len) if blocks else None

def run_code(code, timeout=180):
    with tempfile.TemporaryDirectory() as d:
        p = Path(d) / "solution.py"
        p.write_text(code, encoding="utf-8")
        try:
            r = subprocess.run([sys.executable, str(p)], capture_output=True, text=True, timeout=timeout, cwd=d)
            return r.returncode == 0, r.stdout, r.stderr[-1500:]
        except subprocess.TimeoutExpired:
            return False, "", "TIMEOUT"

d_rows = []
for r in res[res.type == "D"].itertuples():
    code = extract_code(r.response)
    ok, out, err = run_code(code) if code else (False, "", "нет блока кода")
    m = re.findall(r"РЕЗУЛЬТАТ\s*:?\s*(-?\d+(?:\.\d+)?(?:e-?\d+)?)", out)
    value = float(m[-1]) if m else None
    t = task_by_id[r.task_id]
    d_rows.append(dict(task_id=r.task_id, model=r.model, runs=ok, value=value, etalon=t["etalon"],
                       result_ok=bool(ok and value is not None and abs(value - t["etalon"]) <= t["tol"]),
                       stderr=err.strip().splitlines()[-1] if err.strip() else ""))
d_res = pd.DataFrame(d_rows)
d_res.to_csv(WORKDIR / "typeD_runs.csv", index=False)
print(d_res.groupby("model")[["runs", "result_ok"]].mean().rename(columns={"runs": "pass@1", "result_ok": "результат в допуске"}))
d_res

In [ ]:
# Для общей точности: задача D считается решённой, если код запустился и результат в допуске
res = res.merge(d_res[["task_id", "model", "runs", "result_ok"]], on=["task_id", "model"], how="left")
res.loc[res.type == "D", "correct"] = res.loc[res.type == "D", "result_ok"].astype(float)

### 5.2. Слепая ручная проверка B (обоснование), C и D

По таблице метрик задания **балл по рубрике 0–2** ставится для типов B и C. В B вердикт «верно/неверно» уже проверен автоматически, а вручную оценивается обоснование: 0 — вывод неверный; 1 — вывод верный, обоснование с ошибкой; 2 — всё верно. Для D вручную заполняется чек-лист методологических ошибок.

Ячейка ниже создаёт `blind_grading.csv`: ответы перемешаны, название модели скрыто (есть только `blind_id`). Соответствие `blind_id → модель` лежит в отдельном файле `blind_key.csv`, открывать его до оценки нельзя.

**Порядок проверки:** каждый проверяющий копирует файл в `grades_rater1.csv` / `grades_rater2.csv` и заполняет `score` (0/1/2 по рубрике из колонки `рубрика`), а для D ещё колонки чек-листа (1 = ошибка есть). По заданию C и D проверяют **двое независимо и вслепую**: второй участник заполняет свою копию, не видя оценок первого. Ноутбук посчитает каппу Коэна отдельно для C и для D. Если каппа < 0,6, рубрику нужно уточнить и оценить заново.

In [ ]:
D_CHECKLIST = ["scale_before_split", "tuning_on_test", "eval_on_train", "accuracy_on_imbalance", "no_stratify", "no_random_state"]
manual = res[res.type.isin(["B", "C", "D"])][["task_id", "model", "response"]].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
manual["blind_id"] = [f"M{i:03d}" for i in range(len(manual))]
manual[["blind_id", "model"]].to_csv(WORKDIR / "blind_key.csv", index=False)
sheet = manual[["blind_id", "task_id", "response"]].copy()
sheet["задача"] = sheet.task_id.map(lambda i: task_by_id[i]["text_ru"])
RUBRIC = {"B": lambda t: f"эталон: {t['etalon']}. 2: вывод верный и обоснование верное; 1: вывод верный, обоснование с ошибкой; 0: вывод неверный",
          "C": lambda t: t["etalon"],
          "D": lambda t: "2: код корректен методологически; 1: есть 1 ошибка из чек-листа; 0: ≥2 ошибок или не по заданию. "
                         "Чек-лист (1 = ошибка есть): " + ", ".join(D_CHECKLIST)}
sheet["тип"] = sheet.task_id.map(lambda i: task_by_id[i]["type"])
sheet["рубрика"] = sheet.task_id.map(lambda i: RUBRIC[task_by_id[i]["type"]](task_by_id[i]))
sheet["score"] = ""
for c in D_CHECKLIST:
    sheet[c] = ""
if not (WORKDIR / "blind_grading.csv").exists():                 # не перезаписываем, если уже раздали проверяющим
    sheet.to_csv(WORKDIR / "blind_grading.csv", index=False)
print("Слепая таблица:", WORKDIR / "blind_grading.csv", "| строк:", len(sheet))

In [ ]:
def load_grades(name):
    p = WORKDIR / name
    if not p.exists():
        return None
    g = pd.read_csv(p).merge(pd.read_csv(WORKDIR / "blind_key.csv"), on="blind_id")
    return g[g.score.notna() & (g.score.astype(str) != "")]

g1, g2 = load_grades("grades_rater1.csv"), load_grades("grades_rater2.csv")
if g1 is None:
    print("Оценок пока нет: нужны файлы grades_rater1.csv и grades_rater2.csv.")
else:
    g1["score"] = g1.score.astype(int)
    g1["тип"] = g1.task_id.map(lambda i: task_by_id[i]["type"])
    # Доля методологических ошибок в коде (тип D) по чек-листу
    gd = g1[g1["тип"] == "D"].copy()
    if len(gd):
        gd[D_CHECKLIST] = gd[D_CHECKLIST].fillna(0).astype(int)
        gd["есть_ошибка"] = gd[D_CHECKLIST].max(axis=1)
        print("Тип D, доля программ с методологическими ошибками (rater 1):")
        print(gd.groupby("model")[["есть_ошибка"] + D_CHECKLIST].mean().round(2), "\n")
        d_method = gd.groupby("model")["есть_ошибка"].mean()
    if g2 is not None:
        g2 = g2.assign(score=g2.score.astype(int))
        both = g1.merge(g2[["blind_id", "score"]], on="blind_id", suffixes=("_1", "_2"))
        for tp in ["C", "D", "B"]:
            bt = both[both["тип"] == tp]
            if len(bt):
                k = cohen_kappa_score(bt.score_1, bt.score_2)
                print(f"Каппа Коэна, тип {tp} ({len(bt)} ответов): {k:.3f}", "→ уточнить рубрику и переоценить!" if k < 0.6 else "→ согласие достаточное")
        g1 = both.assign(score=(both.score_1 + both.score_2) / 2)
    else:
        print("Только один проверяющий: каппу посчитать нельзя. По заданию C и D должны оценить двое → нужен grades_rater2.csv.")
    manual_scores = g1.groupby(["model", "task_id"]).score.mean().reset_index()
    res = res.merge(manual_scores.rename(columns={"score": "rubric"}), on=["model", "task_id"], how="left")
    print(manual_scores.merge(res[["task_id", "type"]].drop_duplicates(), on="task_id")
          .groupby(["model", "type"]).score.mean().unstack().round(2))

## 6. Тип E: LLM как классификатор

Датасет `load_breast_cancer` встроен в sklearn, скачивать ничего не нужно. Берём 10 «средних» признаков опухоли. **Редкий класс** — злокачественная (≈37%), именно для него считается F1.

- 0, 5 и 20 примеров в промпте (zero-shot / few-shot). Примеры выбираются **стратифицированно** из обучающей части, чтобы в 5 и 20 примерах были оба класса (вопрос самоконтроля № 8).
- 100 тестовых строк, стратифицированных, не пересекаются с примерами.
- Базовые модели: логистическая регрессия и наивный Байес, обученные **на тех же 5 и 20 примерах**, а также константа «всегда частый класс».
- Метрики: F1 для редкого класса и balanced accuracy. Accuracy приводится только для справки.

In [ ]:
from sklearn.naive_bayes import GaussianNB

bc = load_breast_cancer()
FEATS_RU = ["средний радиус", "средняя текстура", "средний периметр", "средняя площадь", "средняя гладкость",
            "средняя компактность", "средняя вогнутость", "среднее число вогнутых точек", "средняя симметрия",
            "средняя фрактальная размерность"]
Xbc, ybc = bc.data[:, :10], bc.target            # 0 = злокачественная (редкий класс), 1 = доброкачественная
LABEL_RU = {0: "злокачественная", 1: "доброкачественная"}
N_TEST = 100
X_pool, X_test_e, y_pool, y_test_e = train_test_split(Xbc, ybc, test_size=N_TEST, stratify=ybc, random_state=RANDOM_STATE)

def pick_shots(k):
    if k == 0:
        return np.empty((0, 10)), np.array([], int)
    Xs, _, ys, _ = train_test_split(X_pool, y_pool, train_size=k, stratify=y_pool, random_state=RANDOM_STATE)
    return Xs, ys

def row_to_text(x):
    return "; ".join(f"{n} = {v:.4g}" for n, v in zip(FEATS_RU, x))

def e_prompt(x, Xs, ys):
    head = ("Опухоль молочной железы описана 10 признаками ядер клеток (по снимку тонкоигольной биопсии). "
            "Определи, злокачественная она или доброкачественная.\n")
    shots = "".join(f"\nПример {i+1}: {row_to_text(a)} → {LABEL_RU[b]}" for i, (a, b) in enumerate(zip(Xs, ys)))
    return (head + (f"\nРазмеченные примеры:{shots}\n" if len(ys) else "") +
            f"\nНовый объект: {row_to_text(x)}\n\nВ строке ОТВЕТ напиши одно слово: злокачественная или доброкачественная.")

def parse_label(text):
    text = text if isinstance(text, str) else ""
    a = (answer_line(text) or text[-200:]).lower()
    if "доброкач" in a:
        return 1
    if "злокач" in a:
        return 0
    return -1                                       # нет ответа = ошибка
SHOTS = {k: pick_shots(k) for k in (0, 5, 20)}
print({k: np.bincount(v[1], minlength=2).tolist() for k, v in SHOTS.items()}, "| тест:", np.bincount(y_test_e).tolist())
print(e_prompt(X_test_e[0], *SHOTS[5])[:900])

In [ ]:
e_rows = []
for k, (Xs, ys) in SHOTS.items():
    for i, x in enumerate(X_test_e):
        for m in MODELS:
            rec = ask_cached(m, e_prompt(x, Xs, ys), key=f"E|k{k}|{i}")
            e_rows.append(dict(model=m, k=k, i=i, y=y_test_e[i], pred=parse_label(rec["response"]),
                               tokens_in=rec["tokens_in"], tokens_out=rec["tokens_out"], seconds=rec["seconds"]))
e_raw = pd.DataFrame(e_rows)
e_raw.to_csv(WORKDIR / "typeE_answers.csv", index=False)
print("Нераспознанных ответов:", (e_raw.pred == -1).sum())

In [ ]:
def e_metrics(y, p):
    return dict(F1_редкий=f1_score(y, p, pos_label=0, labels=[0, 1], zero_division=0),
                bal_acc=balanced_accuracy_score(y, np.where(p == -1, 1 - y, p)),   # нераспознанный = ошибка
                accuracy=accuracy_score(y, p))

e_tab = [dict(model=m, k=k, **e_metrics(g.y.values, g.pred.values)) for (m, k), g in e_raw.groupby(["model", "k"])]
for k in (5, 20):
    Xs, ys = SHOTS[k]
    for name, clf in [("LogReg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))), ("NaiveBayes", GaussianNB())]:
        e_tab.append(dict(model=name, k=k, **e_metrics(y_test_e, clf.fit(Xs, ys).predict(X_test_e))))
majority = np.bincount(y_pool).argmax()
e_tab.append(dict(model="Константа (частый класс)", k=0, **e_metrics(y_test_e, np.full(N_TEST, majority))))
e_tab = pd.DataFrame(e_tab)
display(e_tab.pivot_table(index="model", columns="k", values=["F1_редкий", "bal_acc"]).round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, g in e_tab[e_tab.model != "Константа (частый класс)"].groupby("model"):
    for j, met in enumerate(["F1_редкий", "bal_acc"]):
        ax[j].plot(g.k, g[met], marker="o", label=name, linestyle="--" if name in ("LogReg", "NaiveBayes") else "-")
for j, met in enumerate(["F1_редкий", "bal_acc"]):
    ax[j].axhline(e_tab[e_tab.model.str.startswith("Конст")][met].iloc[0], color="gray", lw=1, label="константа")
    ax[j].set(xlabel="число примеров в обучении/промпте", ylabel=met, xticks=[0, 5, 20], title=f"Тип E: {met}")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(WORKDIR / "typeE.png", dpi=150); plt.show()

## 7. Метрики и значимость

- **95% ДИ бутстрепом**: 1000 раз выбираем задачи с возвращением, считаем точность, берём 2,5-й и 97,5-й перцентили.
- **Тест Макнемара** (точный): для двух моделей на одних и тех же задачах смотрит только на задачи, где модели разошлись (b — первая права, вторая нет; c — наоборот). Точный вариант эквивалентен биномиальному тесту b из (b + c) с p = 0,5.

In [ ]:
from scipy.stats import binomtest

def boot_ci(values, n_boot=1000, seed=RANDOM_STATE, stat=np.mean):
    v = np.asarray(values, float)
    v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    boots = [stat(rng.choice(v, size=len(v), replace=True)) for _ in range(n_boot)]
    return stat(v), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def mcnemar_exact(a, b):
    """a, b — 0/1-векторы правильности двух моделей на одних задачах. Возвращает (b, c, p)."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01, n10 = int(np.sum(a & ~b)), int(np.sum(~a & b))
    if n01 + n10 == 0:
        return n01, n10, 1.0
    try:
        from statsmodels.stats.contingency_tables import mcnemar
        tbl = [[int(np.sum(a & b)), n01], [n10, int(np.sum(~a & ~b))]]
        return n01, n10, float(mcnemar(tbl, exact=True).pvalue)
    except ImportError:                                  # то же самое через биномиальный тест
        return n01, n10, binomtest(n01, n01 + n10, 0.5).pvalue

AUTO = res[res.type.isin(["A", "B", "D", "F"]) & res.correct.notna()]       # задачи с объективной проверкой
acc_rows = []
for m in MODELS:
    est, lo, hi = boot_ci(AUTO[AUTO.model == m].correct)
    acc_rows.append(dict(model=m, accuracy=est, ci_low=lo, ci_high=hi, n=int((AUTO.model == m).sum()),
                         refusal_rate=res[res.model == m].refused.mean()))
acc = pd.DataFrame(acc_rows).set_index("model")
display(acc.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(acc.index, acc.accuracy, yerr=[acc.accuracy - acc.ci_low, acc.ci_high - acc.accuracy], fmt="o", capsize=6)
ax.set(ylim=(0, 1.05), ylabel="точность (A, B, D, F)", title="Точность моделей с 95% бутстреп-ДИ")
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig(WORKDIR / "accuracy_ci.png", dpi=150); plt.show()

In [ ]:
# Тепловая карта «модель × тип задачи»
heat = res[res.type.isin(["A", "B", "D", "F"])].pivot_table(index="model", columns="type", values="correct", aggfunc="mean")
if "rubric" in res:
    heat["C"] = res[res.type == "C"].groupby("model").rubric.mean() / 2               # рубрика 0–2 -> 0–1
heat["E"] = e_tab[e_tab.k == 20].set_index("model").bal_acc                           # balanced accuracy, 20-shot
heat = heat.reindex(index=list(MODELS), columns=[c for c in "ABCDEF" if c in heat.columns])

fig, ax = plt.subplots(figsize=(8, 3.8))
im = ax.imshow(heat.values.astype(float), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns); ax.set_yticks(range(heat.shape[0]), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        ax.text(j, i, "—" if pd.isna(v) else f"{v:.2f}", ha="center", va="center")
plt.colorbar(im, label="доля верных (C: рубрика/2, E: bal. acc 20-shot)")
ax.set_title("Модель × тип задачи"); plt.tight_layout(); plt.savefig(WORKDIR / "heatmap.png", dpi=150); plt.show()

In [ ]:
# Попарные сравнения моделей тестом Макнемара (на одних и тех же задачах A, B, D, F)
wide = AUTO.pivot_table(index="task_id", columns="model", values="correct")
pairs = []
ms = list(MODELS)
for i in range(len(ms)):
    for j in range(i + 1, len(ms)):
        both = wide[[ms[i], ms[j]]].dropna()
        b, c, p = mcnemar_exact(both[ms[i]], both[ms[j]])
        pairs.append(dict(модель_1=ms[i], модель_2=ms[j], точн_1=both[ms[i]].mean(), точн_2=both[ms[j]].mean(),
                          только_1_права=b, только_2_права=c, p_value=p, значимо_005=p < 0.05))
pd.DataFrame(pairs).round(3)

In [ ]:
# Цена и скорость: задержка, токены, стоимость одного правильного ответа
cost = res.groupby("model").agg(latency_s=("seconds", "mean"), tokens_in=("tokens_in", "mean"),
                                tokens_out=("tokens_out", "mean"), n=("task_id", "size"))
def rub(row):
    c = MODELS[row.name]
    if c["price_in"] is None or c["price_out"] is None:
        return np.nan
    return (row.tokens_in * c["price_in"] + row.tokens_out * c["price_out"]) / 1000
cost["rub_per_task"] = cost.apply(rub, axis=1)
cost["accuracy"] = acc.accuracy
cost["rub_per_correct"] = cost.rub_per_task / cost.accuracy
display(cost.round(4))

ok = cost.dropna(subset=["rub_per_correct"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ok.rub_per_task, ok.accuracy, s=80)
for m, r in ok.iterrows():
    ax.annotate(m, (r.rub_per_task, r.accuracy), textcoords="offset points", xytext=(5, 5))
ax.set(xlabel="средняя стоимость задачи, ₽", ylabel="точность", title="Стоимость – точность")
plt.tight_layout(); plt.savefig(WORKDIR / "cost_accuracy.png", dpi=150); plt.show()
if len(ok) < len(cost):
    print("Нет цены для:", list(cost.index.difference(ok.index)), "→ нужны price_in/price_out в MODELS")

In [ ]:
# Локальная модель: время уже есть выше, память берём из Ollama (/api/ps)
try:
    ps = requests.get(f"{OLLAMA_URL}/api/ps", timeout=5).json().get("models", [])
    for p in ps:
        print(f"{p['name']}: в памяти {p.get('size', 0)/2**30:.2f} ГБ, из них в видеопамяти {p.get('size_vram', 0)/2**30:.2f} ГБ")
except Exception as e:
    print("Ollama недоступна:", e)

## 8. Гипотеза H1. Изменённые числа

**Формулировка:** точность на «учебниковых» задачах выше, чем на тех же задачах с изменёнными числами или условием. Большой разрыв означает, что модель скорее «вспоминает», чем решает.

**Как проверяем:** 7 пар (A01–A07 ↔ F01–F07). Для каждой модели считаем точность на оригиналах и на изменённых версиях. Значимость проверяем парным тестом Макнемара: пары «модель × задача», сравниваем правильность оригинала и его изменённой версии. Отдельно стоит пара p6: изменённые числа делают матрицу вырожденной, и правильный ответ становится 1,0. Это проверка, «считает» ли модель или подставляет привычные ответы.

In [ ]:
h1 = res[res.pair.notna()].pivot_table(index=["model", "pair"], columns="kind", values="correct").dropna()
h1_tab = h1.groupby("model").mean()
h1_tab.loc["ВСЕ модели"] = h1.mean()
h1_tab["разрыв"] = h1_tab["учебник"] - h1_tab["изменённая"]
display(h1_tab.round(3))

b, c, p = mcnemar_exact(h1["учебник"], h1["изменённая"])
print(f"Все модели вместе: {len(h1)} пар; правильно только оригинал: {b}, только изменённая: {c}; p = {p:.4f} (Макнемар, точный)")
for m in MODELS:
    hm = h1.loc[m]
    bm, cm, pm = mcnemar_exact(hm["учебник"], hm["изменённая"])
    print(f"  {m:15s}: только оригинал {bm}, только изменённая {cm}, p = {pm:.3f}")

ax = h1_tab.drop(columns="разрыв")[["учебник", "изменённая"]].plot.bar(figsize=(8, 4), rot=15, ylim=(0, 1.05))
ax.set(ylabel="точность", title="H1: учебниковые vs изменённые задачи")
plt.tight_layout(); plt.savefig(WORKDIR / "H1.png", dpi=150); plt.show()

**Как читать результат H1:** гипотеза подтверждается, если разрыв положительный **и** p < 0,05. Если разрыв есть, но p ≥ 0,05, вывод «данных недостаточно»: 7 пар — маленькая выборка, и это ограничение эксперимента. Если разрыв ≈ 0 или отрицательный, гипотеза опровергнута.

## 9. Гипотеза H5. Размер модели

**Формулировка:** маленькая локальная модель отстаёт от больших сильнее в расчётных задачах (A), чем в концептуальных (B).

**Как проверяем:**
- отставание на типе A: gap_A = (средняя точность трёх больших моделей на A) − (точность Qwen3-4B на A);
- то же для B: gap_B;
- разница: Δ = gap_A − gap_B. Гипотеза подтверждается, если Δ > 0 и 95% бутстреп-ДИ для Δ не содержит 0.
  Бутстреп: задачи A и задачи B пересэмплируются отдельно, 1000 раз.

In [ ]:
def gap(sub):
    big = sub[sub.model.isin(BIG_MODELS)].groupby("task_id").correct.mean()
    small = sub[sub.model == SMALL_MODEL].set_index("task_id").correct
    return (big - small).dropna()                     # отставание маленькой модели по каждой задаче

gA, gB = gap(res[res.type == "A"]), gap(res[res.type == "B"])
rng = np.random.default_rng(RANDOM_STATE)
deltas = [rng.choice(gA.values, len(gA)).mean() - rng.choice(gB.values, len(gB)).mean() for _ in range(1000)]
delta = gA.mean() - gB.mean()
h5_lo, h5_hi = np.percentile(deltas, [2.5, 97.5])
print(f"gap_A = {gA.mean():.3f} (n={len(gA)}), gap_B = {gB.mean():.3f} (n={len(gB)})")
print(f"Δ = gap_A − gap_B = {delta:.3f}, 95% ДИ [{h5_lo:.3f}; {h5_hi:.3f}]")

h5 = res[res.type.isin(["A", "B"])].pivot_table(index="model", columns="type", values="correct").reindex(list(MODELS))
ax = h5.plot.bar(figsize=(8, 4), rot=15, ylim=(0, 1.05))
ax.set(ylabel="точность", title="H5: расчётные (A) и концептуальные (B) задачи по моделям")
plt.tight_layout(); plt.savefig(WORKDIR / "H5.png", dpi=150); plt.show()

**Как читать результат H5:** ДИ целиком > 0 — подтверждена; ДИ содержит 0 — данных недостаточно; ДИ целиком < 0 — опровергнута (маленькая модель отстаёт сильнее в концептуальных задачах).

## 10. Гипотеза H10. Калибровка уверенности

**Формулировка:** уверенность, которую модель указывает (0–100), систематически завышена: средняя уверенность выше фактической точности.

**Как проверяем** (на задачах с автоматической проверкой: A, B, F):
- **overconfidence** = средняя уверенность/100 − точность, с бутстреп-ДИ;
- **ECE**: ответы делятся на 5 групп по заявленной уверенности (0–20, 20–40, …, 80–100). В каждой группе сравнивается средняя уверенность с долей верных. ECE = Σ (n_группы / N) · |уверенность − точность|;
- **диаграмма надёжности**: по оси X уверенность, по оси Y доля верных. Идеальная калибровка лежит на диагонали, а точки ниже неё означают завышенную уверенность.

In [ ]:
CAL = res[res.type.isin(["A", "B", "F"]) & res.confidence.notna()].copy()
CAL["conf"] = CAL.confidence / 100
BINS = np.array([0, .2, .4, .6, .8, 1.0])

def ece(conf, correct):
    idx = np.clip(np.digitize(conf, BINS[1:-1], right=True), 0, 4)
    total = 0.0
    for b in range(5):
        mask = idx == b
        if mask.any():
            total += mask.mean() * abs(conf[mask].mean() - correct[mask].mean())
    return total

h10_rows = []
for m, g in CAL.groupby("model"):
    rng = np.random.default_rng(RANDOM_STATE)
    diffs = []
    for _ in range(1000):
        s = g.sample(len(g), replace=True, random_state=int(rng.integers(1e9)))
        diffs.append(s.conf.mean() - s.correct.mean())
    h10_rows.append(dict(model=m, n=len(g), mean_conf=g.conf.mean(), accuracy=g.correct.mean(),
                         overconf=g.conf.mean() - g.correct.mean(), ci_low=np.percentile(diffs, 2.5),
                         ci_high=np.percentile(diffs, 97.5), ECE=ece(g.conf.values, g.correct.values),
                         conf_missing=res[(res.model == m) & res.type.isin(["A", "B", "F"])].confidence.isna().mean()))
h10 = pd.DataFrame(h10_rows).set_index("model").reindex(list(MODELS))
display(h10.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="идеальная калибровка")
for m, g in CAL.groupby("model"):
    idx = np.clip(np.digitize(g.conf, BINS[1:-1], right=True), 0, 4)
    pts = pd.DataFrame(dict(b=idx, conf=g.conf.values, cor=g.correct.values)).groupby("b").agg(conf=("conf", "mean"), cor=("cor", "mean"), n=("cor", "size"))
    ax[0].plot(pts.conf, pts.cor, marker="o", label=m)
    for _, r in pts.iterrows():
        ax[0].annotate(int(r.n), (r.conf, r.cor), fontsize=7, xytext=(3, -8), textcoords="offset points")
ax[0].set(xlabel="заявленная уверенность", ylabel="доля верных", title="Диаграмма надёжности (числа = кол-во ответов)",
          xlim=(0, 1.02), ylim=(0, 1.05)); ax[0].legend(fontsize=8)
ax[1].errorbar(h10.index, h10.overconf, yerr=[h10.overconf - h10.ci_low, h10.ci_high - h10.overconf], fmt="o", capsize=6)
ax[1].axhline(0, color="gray", lw=1)
ax[1].set(ylabel="ср. уверенность − точность", title="H10: завышение уверенности (95% ДИ)")
plt.setp(ax[1].get_xticklabels(), rotation=15)
plt.tight_layout(); plt.savefig(WORKDIR / "H10.png", dpi=150); plt.show()

**Как читать результат H10:** если у модели ДИ для «уверенность − точность» целиком > 0, у неё уверенность завышена. Гипотеза «систематически завышена» подтверждается, если так у большинства моделей. Если модель часто не пишет уверенность (`conf_missing`), это нужно указать в отчёте.

## 11. Ошибки для разбора

Здесь собраны все неверные ответы на задачи с автоматической проверкой. Из них отбираются 5 самых показательных для разбора в отчёте: в каком месте рассуждения модель сбилась и почему. Полный текст ответа есть в `raw_answers.csv`.

In [ ]:
errors = res[(res.correct == 0) & res.type.isin(["A", "B", "F"])][["task_id", "model", "etalon", "parsed", "confidence", "refused", "response"]]
errors = errors.assign(etalon=errors.etalon.map(lambda v: round(v, 4) if isinstance(v, float) else v))
errors.to_csv(WORKDIR / "errors.csv", index=False)
print("Всего ошибок:", len(errors))
errors.drop(columns="response").head(40)

In [ ]:
# Сводка по гипотезам: итог по заранее заданным критериям
def verdict(effect, lo, hi):
    if lo > 0: return "подтверждена"
    if hi < 0: return "опровергнута"
    return "данных недостаточно" if effect > 0 else "не подтвердилась"

h1_p = mcnemar_exact(h1["учебник"], h1["изменённая"])[2]
h1_gap = h1_tab.loc["ВСЕ модели", "разрыв"]
summary = pd.DataFrame([
    dict(гипотеза="H1 изменённые числа", прогноз="подтвердится", эффект=round(h1_gap, 3), детали=f"p (Макнемар) = {h1_p:.3f}",
         итог=("подтверждена" if (h1_gap > 0 and h1_p < 0.05) else "данных недостаточно" if h1_gap > 0
              else "опровергнута (обратный эффект)" if h1_p < 0.05 else "не подтвердилась (разрыва нет)")),
    dict(гипотеза="H5 размер модели", прогноз="подтвердится", эффект=round(delta, 3), детали=f"95% ДИ [{h5_lo:.3f}; {h5_hi:.3f}]",
         итог=verdict(delta, h5_lo, h5_hi)),
    dict(гипотеза="H10 калибровка", прогноз="подтвердится", эффект=round(h10.overconf.mean(), 3),
         детали=f"завышена у {(h10.ci_low > 0).sum()} из {len(h10)} моделей; ECE: " + ", ".join(f"{m}={v:.2f}" for m, v in h10.ECE.items()),
         итог="подтверждена" if (h10.ci_low > 0).sum() > len(h10) / 2 else "не подтвердилась"),
])
summary.to_csv(WORKDIR / "hypotheses_summary.csv", index=False)
summary